# 10 · The 344 genes of the STAMP model

This notebook builds a second X with the **344 genes of the published STAMP breast model** (Ben Cohen et al. 2025): TP53 and its neighbors in the HumanNet v2 graph. Notebooks 11 and 12 use it.

- **Notebook 12** applies the STAMP model, with its published weights, to CPTAC and SMC. It needs the 344 genes in the order of the model, **TP53 included**, because the weights were trained with them.
- **Notebook 11** trains our MLP with the same genes **without TP53** (343), as an extra analysis. TP53 stays out by the rule of 02-Oct-2026.

Our main X (notebook 02) has 340 of the 344 genes. The other 4 are not there for different reasons:

| Gene | Why it is not in the main X |
|---|---|
| TP53 | Left out on purpose (02-Oct) |
| INGX | HGNC classifies it as a pseudogene, and X only has protein-coding genes |
| TAX1BP3 | Not measured by the METABRIC microarray, so it is not common to the 4 cohorts |
| SOD2 | Not in the GDC TPM matrices of cBioPortal (TCGA and CPTAC) |

So the genes are read again from the raw files, with the same steps as notebook 02: genes joined by Entrez ID, the row with the highest mean expression when a gene has more than one row (Miller et al. 2011), and log2(TPM + 1) for RNA-seq. A gene that a cohort does not have stays empty. `common.zscore` turns it into 0 after the z-score, which is what STAMP did with the genes missing in METABRIC (decision of 29-Oct-2026).

**Input:** the STAMP model file (`data/raw/stamp/`, downloaded from the STAMP GitHub if missing), the raw expression files, `hgnc_complete_set.txt` and the main X (for the tumors and to check the values).
**Output:** `data/processed/X_stamp_<cohort>.parquet` (tumors × 344 genes, with the gene symbols of STAMP as column names).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
from common import DATA, FILES, cargar, cargar_stamp

RAW = ROOT / "data" / "raw"
COHORTS = {"TCGA-BRCA": ("brca_tcga_gdc", "data_mrna_seq_tpm.txt", True),
           "METABRIC": ("brca_metabric", "data_mrna_illumina_microarray.txt", False),
           "CPTAC": ("breast_cptac_gdc", "data_mrna_seq_tpm.txt", True),
           "SMC 2018": ("brca_smc_2018", "data_mrna_seq_tpm.txt", True)}  # (folder, file, is RNA-seq)

## Step 1 · The genes of the model

The `.pt` file keeps the expression table it was trained with. Its columns are the 344 genes, in the order the network expects.

In [2]:
stamp = cargar_stamp()
genes = list(stamp.X.columns)
print(len(genes), "genes · TP53 included:", "TP53" in genes)

344 genes · TP53 included: True


## Step 2 · From gene symbol to Entrez ID

Our data are joined by Entrez ID. Each STAMP symbol is looked up in HGNC, first as the current symbol and then as a previous symbol. One gene has an old name in STAMP: FAM173A, now ANTKMT.

In [3]:
hgnc = pd.read_csv(RAW / "referencias" / "hgnc_complete_set.txt", sep="\t", low_memory=False,
                   usecols=["symbol", "prev_symbol", "entrez_id"]).dropna(subset=["entrez_id"])
current = dict(zip(hgnc.symbol, hgnc.entrez_id.astype(int)))
previous = hgnc.assign(prev_symbol=hgnc.prev_symbol.str.split("|")).explode("prev_symbol").dropna(subset=["prev_symbol"])
previous = previous.drop_duplicates("prev_symbol", keep=False)   # an old name shared by two genes is not used
previous = dict(zip(previous.prev_symbol, previous.entrez_id.astype(int)))

entrez = pd.Series({g: current.get(g, previous.get(g)) for g in genes})
assert entrez.notna().all() and entrez.is_unique, entrez[entrez.isna() | entrez.duplicated(keep=False)]
entrez = entrez.astype(int)
print("Old names:", {g: e for g, e in entrez.items() if g not in current})

Old names: {'FAM173A': 65990}


## Step 3 · Read the 344 genes in each cohort

Same tumors and same order as the main X. For the 340 genes that are also in the main X, the values must be the same. That checks that the steps are the ones of notebook 02.

In [4]:
smc_names = pd.read_csv(DATA / "revision" / "smc_traduccion_nombres.csv")
missing = {}
for name, (folder, file, rnaseq) in COHORTS.items():
    x_main, _ = cargar(name)
    e = pd.read_csv(RAW / folder / file, sep="\t", low_memory=False,
                    usecols=lambda c: c in set(x_main.index) | {"Hugo_Symbol", "Entrez_Gene_Id"})
    if e.Entrez_Gene_Id.isna().all():   # SMC has no Entrez IDs: translated by name, as in notebook 02
        e["Entrez_Gene_Id"] = e.Hugo_Symbol.map(dict(zip(smc_names.hugo_symbol_smc, smc_names.entrez)))
    e = e.dropna(subset=["Entrez_Gene_Id"]).drop(columns="Hugo_Symbol", errors="ignore")
    e = e.set_index(e.pop("Entrez_Gene_Id").astype(int))
    e = e[e.index.isin(entrez)]
    e = e.loc[e.mean(axis=1).sort_values(ascending=False).index]
    e = e[~e.index.duplicated()]   # the row with the highest mean expression
    x = e.reindex(entrez.values).T.loc[x_main.index]
    x.columns = genes
    if rnaseq:
        x = np.log2(x + 1)

    shared = [g for g in genes if str(entrez[g]) in x_main.columns]
    assert np.allclose(x[shared], x_main[[str(entrez[g]) for g in shared]], atol=1e-4, equal_nan=True), name
    x.astype("float32").to_parquet(DATA / f"X_stamp_{FILES[name]}.parquet")
    missing[name] = [g for g in genes if x[g].isna().all()]
    print(f"{name}: {x.shape[0]} tumors · {len(shared)} genes equal to the main X · missing: {missing[name]}")

TCGA-BRCA: 961 tumors · 340 genes equal to the main X · missing: ['SOD2']


METABRIC: 1899 tumors · 340 genes equal to the main X · missing: ['TAX1BP3']


CPTAC: 105 tumors · 340 genes equal to the main X · missing: ['SOD2']


SMC 2018: 167 tumors · 340 genes equal to the main X · missing: []


Expected: SOD2 missing in TCGA and CPTAC, and TAX1BP3 missing in METABRIC. SMC has all 344.